# Peek at KO/EC links — annotation rows ↔ biosamples / studies

Worked-example queries showing how to navigate from a row in
`nmdc_results.annotation_kegg_orthology` (or `annotation_enzyme_commission`)
out to the biosample, study, and file metadata in `nmdc_metadata`.

**Anchor columns on the annotation tables:**

| column | meaning | join target |
|---|---|---|
| `workflow_run_id` | `nmdc:wfmgan-...` MetagenomeAnnotation activity | `nmdc_metadata.workflow_execution_set.id` |
| `data_object_id`  | `nmdc:dobj-...` source TSV file | `nmdc_metadata.data_object_set.id` |
| `gene_id`         | `<workflow_run_id>_<contig>_<start>_<end>` | local; matches functional GFFs (issues #84, #86–#90) |
| `ncbi_taxid`      | NCBI taxon for the BLAST hit | external (no native BERDL taxonomy table) |
| `annotation_id`   | `KO:Kxxxxx` or `EC:n.n.n.n` | KEGG / EC ontologies |

**The chain:**

```
annotation_*.workflow_run_id
  → workflow_execution_set.id  (.was_informed_by is array of dgns IDs)
  → data_generation_set.id
    → data_generation_set_has_input.has_input            (biosample IDs)
    → data_generation_set_associated_studies.associated_studies
  → biosample_set.{env_*, geo_loc_name_*, depth, …}
```

### Session setup

In [13]:
spark = get_spark_session(app_name="peek_ko_ec_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

Spark version: 4.0.1


### Preflight: confirm the annotation tables are registered

In [14]:
registered = {r.tableName for r in spark.sql("SHOW TABLES IN nmdc_results").collect()}
for tbl in ("annotation_kegg_orthology", "annotation_enzyme_commission"):
    if tbl not in registered:
        raise RuntimeError(f"nmdc_results.{tbl} not registered — run ingest_ko_ec_annotations.ipynb first")
    print(f"OK: nmdc_results.{tbl}")

OK: nmdc_results.annotation_kegg_orthology
OK: nmdc_results.annotation_enzyme_commission


### 1. Pick an anchor row

Grab one annotation row to drive the rest of the joins. Same recipe works for EC.

In [15]:
_anchor_df = spark.sql("""
    SELECT gene_id, annotation_id, workflow_run_id, data_object_id, ncbi_taxid
    FROM   nmdc_results.annotation_kegg_orthology
    WHERE  workflow_run_id IS NOT NULL
      AND  data_object_id  IS NOT NULL
      AND  annotation_id   IS NOT NULL
    LIMIT 1
""").toPandas()
if _anchor_df.empty:
    raise RuntimeError("no row in nmdc_results.annotation_kegg_orthology has non-null workflow_run_id/data_object_id/annotation_id")
anchor = _anchor_df.iloc[0]
WORKFLOW_RUN_ID = anchor["workflow_run_id"]
DATA_OBJECT_ID  = anchor["data_object_id"]
ANNOTATION_ID   = anchor["annotation_id"]
print(anchor.to_string())

gene_id            nmdc:wfmgan-11-kepa2m52.1_083196_3_293
annotation_id                                   KO:K21449
workflow_run_id                 nmdc:wfmgan-11-kepa2m52.1
data_object_id                      nmdc:dobj-11-r1d6nr97
ncbi_taxid                                     2883257780


### 2. `workflow_run_id` → workflow_execution_set

`was_informed_by` is array-typed on `workflow_execution_set` (a workflow run
can be informed by multiple data generations). The Silver layer flattens
this into `nmdc_metadata.workflow_execution_set_was_informed_by` (one row
per `(parent_id, was_informed_by)` pair) — prefer this side table over
`EXPLODE` for downstream joins.

In [16]:
spark.sql(f"""
    SELECT id, type, was_informed_by, started_at_time, ended_at_time
    FROM nmdc_metadata.workflow_execution_set
    WHERE id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,id,type,was_informed_by,started_at_time,ended_at_time
0,nmdc:wfmgan-11-kepa2m52.1,nmdc:MetagenomeAnnotation,[nmdc:dgns-11-wm0baw87],2025-06-10 12:57:50,NaN


### 3. workflow_execution → data_generation_set (the sequencing run)

In [17]:
spark.sql(f"""
    SELECT dg.id, dg.type, dg.name, dg.processing_institution,
           dg.principal_investigator_name, dg.start_date, dg.end_date
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by wib
    JOIN   nmdc_metadata.data_generation_set dg
             ON dg.id = wib.was_informed_by
    WHERE  wib.parent_id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,id,type,name,processing_institution,principal_investigator_name,start_date,end_date
0,nmdc:dgns-11-wm0baw87,nmdc:NucleotideSequencing,Terrestrial soil microbial communities - ONAQ_...,Battelle,NaN,NaN,NaN


### Aside: provenance graph and `biosample_to_workflow_run`

NMDC's data model has two parallel chains, bridged by `data_generation`:

```
biosample → material_processing → procsm → … → data_generation
                                                       ↓
                              data_object ← workflow_execution
```

Walking this graph from a workflow run back to source biosamples was previously
done with a Trino `WITH RECURSIVE` CTE (which hits a 150-stage planner limit
at scale) or an iterative Python BFS. Both are now replaced by a single JOIN
to the precomputed `nmdc_metadata.biosample_to_workflow_run` table.

That table was built by `notebooks/build_biosample_to_workflow_run.ipynb`
and must be rebuilt after each NMDC data load.

### 4. workflow_execution → biosample(s)

A direct JOIN to `biosample_to_workflow_run` replaces the recursive graph
walk. The precomputed table covers any chain depth and records which
MaterialProcessing classes appeared in the provenance chain.

In [ ]:
spark.sql(f"""
    SELECT b2wr.biosample_id, b2wr.n_hops,
           bsm.env_broad_scale_term_id, bsm.env_local_scale_term_id,
           bsm.env_medium_term_id, bsm.geo_loc_name_has_raw_value,
           bsm.depth_has_numeric_value, bsm.depth_has_unit
    FROM   nmdc_metadata.biosample_to_workflow_run b2wr
    JOIN   nmdc_metadata.biosample_set bsm ON bsm.id = b2wr.biosample_id
    WHERE  b2wr.workflow_run_id = '{WORKFLOW_RUN_ID}'
""").toPandas()

### 5. data_generation → study

In [19]:
spark.sql(f"""
    SELECT s.id AS study_id, s.name, s.title, s.principal_investigator_name
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by wib
    JOIN   nmdc_metadata.data_generation_set_associated_studies dgs
             ON dgs.parent_id = wib.was_informed_by
    JOIN   nmdc_metadata.study_set s
             ON s.id = dgs.associated_studies
    WHERE  wib.parent_id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,study_id,name,title,principal_investigator_name
0,nmdc:sty-11-34xj1150,National Ecological Observatory Network: soil ...,National Ecological Observatory Network: soil ...,Kate Thibault


### 6. `data_object_id` → data_object_set (file-level provenance)

Useful for verifying which source TSV a row came from, file size, MD5,
URL, etc. Not needed for downstream science.

In [20]:
spark.sql(f"""
    SELECT id, name, data_object_type, file_size_bytes, md5_checksum, url
    FROM nmdc_metadata.data_object_set
    WHERE id = '{DATA_OBJECT_ID}'
""").toPandas()

,id,name,data_object_type,file_size_bytes,md5_checksum,url
0,nmdc:dobj-11-r1d6nr97,nmdc_wfmgan-11-kepa2m52.1_ko.tsv,Annotation KEGG Orthology,5100559,5837e9884f679f1cabaadaab378dc771,https://data.microbiomedata.org/data/nmdc:dgns...


### 7. Cross-check against `functional_annotation_agg`

`functional_annotation_agg` is a precomputed `(workflow_run, gene_function_id, count)`
rollup. Two caveats when joining to it:

1. **EC is absent** — `functional_annotation_agg` only carries `PFAM`,
   `KEGG.ORTHOLOGY`, and `COG`. The new `annotation_enzyme_commission` table
   is the only source of EC in BERDL.
2. **KO prefix differs** — `KO:Kxxxxx` here vs `KEGG.ORTHOLOGY:Kxxxxx` there.
   Translate before joining. The two values per workflow run *should* line up
   (`COUNT(*)` in `annotation_kegg_orthology` ≈ `SUM(count)` in the agg);
   any drift is worth flagging.

In [21]:
spark.sql(f"""
    WITH ours AS (
        SELECT 'KEGG.ORTHOLOGY:' || SUBSTRING(annotation_id, 4) AS gene_function_id,
               COUNT(*) AS hits_in_ko_table
        FROM nmdc_results.annotation_kegg_orthology
        WHERE workflow_run_id = '{WORKFLOW_RUN_ID}'
        GROUP BY 1
    ),
    theirs AS (
        SELECT gene_function_id, count AS count_in_agg
        FROM nmdc_metadata.functional_annotation_agg
        WHERE was_generated_by = '{WORKFLOW_RUN_ID}'
          AND gene_function_id LIKE 'KEGG.ORTHOLOGY:%'
    )
    SELECT COALESCE(ours.gene_function_id, theirs.gene_function_id) AS gene_function_id,
           ours.hits_in_ko_table,
           theirs.count_in_agg,
           COALESCE(ours.hits_in_ko_table, 0) - COALESCE(theirs.count_in_agg, 0) AS diff
    FROM ours FULL OUTER JOIN theirs USING (gene_function_id)
    ORDER BY ABS(COALESCE(ours.hits_in_ko_table, 0) - COALESCE(theirs.count_in_agg, 0)) DESC
    LIMIT 20
""").toPandas()

,gene_function_id,hits_in_ko_table,count_in_agg,diff
0,KEGG.ORTHOLOGY:K00001,14,14,0
1,KEGG.ORTHOLOGY:K00003,32,32,0
2,KEGG.ORTHOLOGY:K00004,5,5,0
3,KEGG.ORTHOLOGY:K00008,29,29,0
4,KEGG.ORTHOLOGY:K00009,2,2,0
5,KEGG.ORTHOLOGY:K00010,19,19,0
6,KEGG.ORTHOLOGY:K00012,44,44,0
7,KEGG.ORTHOLOGY:K00013,30,30,0
8,KEGG.ORTHOLOGY:K00014,24,24,0
9,KEGG.ORTHOLOGY:K00015,13,13,0


### 8. End-to-end: KO hits per biosample for a chosen KO

A single Spark SQL query joins the annotation table to
`biosample_to_workflow_run` — no recursive walk, no Trino cursor.
The precomputed provenance bridge does the graph traversal for free.

In [ ]:
import time
TARGET_KO = ANNOTATION_ID  # substitute any 'KO:Kxxxxx'

t0 = time.monotonic()
result = spark.sql(f"""
    SELECT b2wr.biosample_id,
           bsm.env_broad_scale_term_id,
           bsm.env_medium_term_id,
           bsm.geo_loc_name_has_raw_value,
           SUM(ko.n_hits) AS total_hits
    FROM (
        SELECT workflow_run_id, COUNT(*) AS n_hits
        FROM   nmdc_results.annotation_kegg_orthology
        WHERE  annotation_id = '{TARGET_KO}'
        GROUP  BY workflow_run_id
    ) ko
    JOIN nmdc_metadata.biosample_to_workflow_run b2wr
      ON b2wr.workflow_run_id = ko.workflow_run_id
    JOIN nmdc_metadata.biosample_set bsm
      ON bsm.id = b2wr.biosample_id
    GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id,
             bsm.env_medium_term_id, bsm.geo_loc_name_has_raw_value
    ORDER BY total_hits DESC
    LIMIT 20
""").toPandas()
print(f"{len(result)} biosamples  ({time.monotonic()-t0:.1f}s)")
result

### 9. Annotation run QC — `annotation_statistics`

`nmdc_results.annotation_statistics` has one row per annotation workflow run with 17 QC
metrics: sequence counts, gene-type counts (CDS, tRNA, ncRNA, rRNA, CRISPR), coding
density, and genes-per-Mbp. Join to biosamples the same way as any other results table.

In [ ]:
if "annotation_statistics" not in {r.tableName for r in spark.sql("SHOW TABLES IN nmdc_results").collect()}:
    print("SKIP: annotation_statistics not registered")
else:
    # QC metrics for the anchor workflow run
    display(spark.sql(f"""
        SELECT ann.workflow_run_id,
               ann.n_seqs, ann.n_cds, ann.n_trna, ann.n_rrna,
               ann.coding_density_pct, ann.genes_per_1m_bp,
               bsm.id AS biosample_id, bsm.env_broad_scale_term_id
        FROM   nmdc_results.annotation_statistics ann
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = ann.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  ann.workflow_run_id = '{WORKFLOW_RUN_ID}'
    """).toPandas())

#### Distribution across biosamples

Coding density and gene content across all annotation runs, grouped by broad-scale environment.

In [ ]:
if "annotation_statistics" not in {r.tableName for r in spark.sql("SHOW TABLES IN nmdc_results").collect()}:
    print("SKIP: annotation_statistics not registered")
else:
    display(spark.sql("""
        SELECT bsm.env_broad_scale_term_id,
               COUNT(DISTINCT b2wr.biosample_id)  AS n_biosamples,
               ROUND(AVG(ann.coding_density_pct), 2) AS avg_coding_density,
               ROUND(AVG(ann.genes_per_1m_bp), 0)    AS avg_genes_per_mbp,
               ROUND(AVG(ann.n_cds), 0)               AS avg_n_cds
        FROM   nmdc_results.annotation_statistics ann
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = ann.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        GROUP BY bsm.env_broad_scale_term_id
        ORDER BY n_biosamples DESC
        LIMIT 20
    """).toPandas())